In [ ]:
import json
from typing import Literal
from typing_extensions import NotRequired

from langchain.messages import AIMessage, ToolMessage, HumanMessage
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.agents import AgentState, create_agent
from langgraph.graph import StateGraph, START, END

In [ ]:
class MultiAgentState(AgentState):
    active_agent: NotRequired[str]

In [ ]:
@tool
def transfer_to_summarise(
    runtime: ToolRuntime,
) -> Command:
    """Transfer to Summarise Agent."""
    latest_ai_message = next(
        msg for msg in reversed(runtime.state["messages"])
        if isinstance(msg, AIMessage))
    transfer_message = ToolMessage(
        content = "Transferred to Summarise Agent",
        tool_call_id = runtime.tool_call_id,
    )
    return Command(
        goto = "summarise_agent",
        update = {
            "active_agent": "summarise_agent",
            "messages": [latest_ai_message, transfer_message],
        },
        graph = Command.PARENT
    )

In [ ]:
@tool
def transfer_to_translate(
    runtime:ToolRuntime,
)-> Command:
    "Transfer to Translate Agent"
    latest_ai_message = next(
        msg for msg in reversed(runtime.state["messages"]) 
        if isinstance(msg, AIMessage)
    )
    transfer_message = ToolMessage(
        content = "Transfer to Translate Agent",
        tool_call_id = runtime.tool_call_id,
    )
    print(f"running transfer to translate tool")
    return Command(
        goto ="translate_agent",
        update = {
            "active_agent":"translate_agent",
            "messages": [latest_ai_message, transfer_message],
        },
        graph = Command.PARENT
    )

In [ ]:
@tool
def transfer_to_email(
    runtime:ToolRuntime,
)-> Command:
    "Transfer to Email Agent"
    latest_ai_message = next(
        msg for msg in reversed(runtime.state["messages"]) 
        if isinstance(msg, AIMessage)
    )
    transfer_message = ToolMessage(
        content = "Transfer to Email Agent",
        tool_call_id = runtime.tool_call_id,
    )
    return Command(
        goto ="email_agent",
        update = {
            "active_agent":"email_agent",
            "messages": [latest_ai_message, transfer_message],
        },
        graph = Command.PARENT
    )

In [ ]:
orchestrator_agent = create_agent(
    model="gpt-4o-mini",
    system_prompt="""You are an orchestrator agent. Based on the user input that is provided to you, you have to determine which of the sub-agent to call:
    1. email_agent: An agent that handles all the email related query
    1. summarise_agent: An agent that handles summairsation related query
    1. translate_agent: An agent that handles language translation related query.
    Just give one word answer on the name of the agent to call and nothing else.
    """,
)

summarise_agent = create_agent(
    model="gpt-4o-mini",
    system_prompt="""You are a summariser agent. You provide output in json format with the keys:
    summarised_text: 
    next_agent: 
    You have to provide a summarised text of the user requested data and
    based on user query, you have to determine the path of agent: There are three possible values for next agent:
    1. translate_agent
    2. email_agent
    3. __end__
    fill the next agent key with only these three possible values. Provide a clean json structure without any json tags in the response.

    """,
)

fallback_agent = create_agent(
    model="gpt-4o-mini",
    system_prompt="You are a fallback agent. When other sub-agents couldn't handle the request, fallback happens to you. So handle the request accordingly" ,
)
translate_agent = create_agent(
    model="gpt-4o-mini",
    system_prompt="""You are a translation agent. Help with language translation inquiries. You provide output in json format with the keys:
    summarised_text: 
    next_agent: 
    You have to provide a summarised text of the user requested data and
    based on user query, you have to determine the path of agent: There are three possible values for next agent:
    2. email_agent
    3. __end__
    fill the next agent key with only these three possible values. Provide a clean json structure without any json tags in the response.
    Only call or transfer_to_email ONCE, even if multiple target languages are requested.""",
)

email_agent = create_agent(
    model="gpt-4o-mini",
    tools=[],
    system_prompt="You are a email agent. Help with drafting, sending email to the user. If the user has asked about sending the email, use human-in-the-loop for confirmation.",
)

In [ ]:
def call_orchestrator_agent(state: MultiAgentState):
    """Node that calls the summarise_agent."""
    response = orchestrator_agent.invoke(state)
    print("after calling the orchestrator agent the output is:",response["messages"][-1].content)
    state['active_agent'] = response["messages"][-1].content
    print(state['active_agent'])
    return response


def call_summarise_agent(state: MultiAgentState):
    """Node that calls the summarise_agent."""
    print("calling summarize agent")
    response = summarise_agent.invoke(state)
    print(f"Output from summairse agent: {response}")
    return response

def call_fallback_agent(state: MultiAgentState):
    """Node that calls the fallback."""
    print("in fallback agent node")
    response = fallback_agent.invoke(state)
    return response


def call_translate_agent(state: MultiAgentState):
    print("invoking translate")
    """Node that calls the translate_agent."""
    response = translate_agent.invoke(state)
    return response

def call_email_agent(state: MultiAgentState):
    """Node that calls the email agent."""
    print("invoking email")
    response = email_agent.invoke(state)
    return response

In [ ]:
def route_after_translate_agent(
    state: MultiAgentState
    )-> Literal[
        "summarise_agent", 
        "translate_agent", 
        "email_agent",
        "fallback_agent",
        "__end__"
        ]:
    """Route based on active_agent, or END if the agent finished without handoff."""
    messages = json.loads(state['messages'][-1].content)
    state['messages'] =  messages['summarised_text']
    next_agent = messages['next_agent']
    print(f"translationagent agent output is:{next_agent}")
    return next_agent


def route_after_agent(
    state: MultiAgentState
    )-> Literal[
        "summarise_agent", 
        "translate_agent", 
        "email_agent",
        "fallback_agent",
        "__end__"
        ]:
    """Route based on active_agent, or END if the agent finished without handoff."""
    messages = state.get("messages", [])
    if messages:
        last_msg = messages[-1]
        if isinstance(last_msg, AIMessage) and not last_msg.tool_calls:
            return "__end__"
    active = state.get("active_agent")
    print(active)
    return active if active else "fallback_agent"


def route_after_summarise_agent(
    state: MultiAgentState
    )-> Literal[
        "summarise_agent", 
        "translate_agent", 
        "email_agent",
        "fallback_agent",
        "__end__"
        ]:
    """Route based on active_agent, or END if the agent finished without handoff."""
    messages = json.loads(state['messages'][-1].content)
    state['messages'] =  messages['summarised_text']
    next_agent = messages['next_agent']
    print(f"Summarizer agent output is:{next_agent}")

    return next_agent


In [ ]:
{'summarised_text': 'Artificial intelligence (AI) is a computer science field that develops systems able to perform tasks that require human intelligence like learning and problem-solving. It enhances modern technology through machine learning and neural networks, widely used in generative tools, predictive analysis, and automation.',
 'next_agent': 'translate_agent'}

In [ ]:
def route_initial(
        state: MultiAgentState
) -> Literal[
        "summarise_agent",
        "translate_agent",
        "email_agent",
        ]:
    print("By orchestrator agent, GOTO:",state['messages'][-1].content)
    return state['messages'][-1].content or "fallback_agent"


In [ ]:
builder = StateGraph(MultiAgentState)
builder.add_node("orchestrator_agent", call_orchestrator_agent)
builder.add_node("email_agent", call_email_agent)
builder.add_node("fallback_agent", call_fallback_agent)
builder.add_node("translate_agent", call_translate_agent)
builder.add_node("summarise_agent", call_summarise_agent)

In [ ]:
builder.add_edge("__start__", "orchestrator_agent")

In [ ]:
builder.add_conditional_edges("orchestrator_agent", route_initial, ["summarise_agent", "translate_agent", "fallback_agent","email_agent", "__end__"])

# After each agent, check if we should end or route to another agent
builder.add_conditional_edges(
    "summarise_agent", route_after_summarise_agent, ["translate_agent",  "email_agent", END]
)
builder.add_conditional_edges(
    "translate_agent", route_after_translate_agent, ["email_agent", END]
)

In [ ]:
graph = builder.compile()


In [ ]:
graph

In [ ]:
text = """Artificial intelligence (AI) is a branch of computer science that creates systems capable of performing tasks requiring human intelligence, such as learning, reasoning, and problem-solving. It powers modern technology through machine learning and neural networks, widely used for generative tools, predictive analysis, and automation in daily life and business."""
result = graph.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": f"Hi, I want you to summairze the given text and then translate it to nepali and hindi: {text}",
            }
        ]
    }
)


In [ ]:
import json
result['messages'][-1].content

In [ ]:
print(result['messages'][-1].content)

In [ ]:
ParentCommand: Command(update={'active_agent': 'translate_agent', 'messages': [AIMessage(content='Artificial intelligence (AI) is a field within computer science that develops systems to perform tasks that typically require human intelligence, such as learning, reasoning, and problem-solving. AI enhances modern technology using machine learning and neural networks, and it is commonly applied in generative tools, predictive analysis, and automation in both daily life and business.\n\nNow, I will translate the summary into Nepali and Hindi. ', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 122, 'prompt_tokens': 199, 'total_tokens': 321, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_43b64008fd', 'id': 'chatcmpl-DclICou3tcjQfxxOmyH95woZB9jAE', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e00d8-2d7c-74d2-af76-2d1753c00f8e-0', tool_calls=[{'name': 'transfer_to_translate', 'args': {}, 'id': 'call_WSOFLMCar2oZUKyMEYf4hVv4', 'type': 'tool_call'}, {'name': 'transfer_to_translate', 'args': {}, 'id': 'call_Qg7T3WuMWd43eDJjo8irvhw0', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 199, 'output_tokens': 122, 'total_tokens': 321, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}), ToolMessage(content='Transfer to Translate Agent', name='transfer_to_translate', tool_call_id='call_Qg7T3WuMWd43eDJjo8irvhw0')]}, goto='translate_agent')


In [ ]:
result["messages"][-1].content

In [ ]:
print(result["messages"][-1].content)

In [ ]:
result

In [ ]:

from langchain_agent.schemas.agent_schemas.graph_input_schema import GraphInput
class GraphBuilder:
    def __init__(self):
        self.builder = StateGraph(MultiAgentState)
        self.graph = None
        self._build()

    def _build(self):
        self._build_nodes()
        self._build_edges()
        self.graph = self.builder.compile()

    def get_graph(self):
        return self.graph

    def run(self, user_input: GraphInput):
        if self.graph is None:
            raise RuntimeError("Graph is not built. Please build the graph first.")
        self.graph.invoke(user_input)

    def _build_nodes(self):
        self.builder.add_node("orchestrator_agent", call_orchestrator_agent)
        self.builder.add_node("email_agent", call_email_agent)
        self.builder.add_node("fallback_agent", call_fallback_agent)
        self.builder.add_node("translate_agent", call_translate_agent)
        self.builder.add_node("summarise_agent", call_summarise_agent)

    def _build_edges(self):
        builder.add_edge(
            "__start__",
            "orchestrator_agent"
        )
        builder.add_conditional_edges(
            "orchestrator_agent", 
            route_initial, 
            [
                "summarise_agent", 
                "translate_agent", 
                "fallback_agent", 
                "email_agent", 
                "__end__"
            ]
        )
        builder.add_conditional_edges(
            "summarise_agent", 
            route_after_summarise_agent, 
            [
                "translate_agent", 
                "email_agent", 
                END
            ]
        )
        builder.add_conditional_edges(
            "translate_agent", 
            route_after_agent, [
                "email_agent", 
                END
            ]
        )


In [ ]:
result

In [ ]:
{'response': [HumanMessage(content='Hi, I want you to summairze the given text and then translate it to nepali and hindi: Artificial intelligence (AI) is a branch of computer science that creates systems capable of performing tasks requiring human intelligence, such as learning, reasoning, and problem-solving. It powers modern technology through machine learning and neural networks, widely used for generative tools, predictive analysis, and automation in daily life and business. and then mail the content to the user ukpaajee@gmail.com', additional_kwargs={}, response_metadata={}, id='ca0dbdaf-6e79-48ba-8d60-fb20ef786908'), AIMessage(content='email_agent', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 2, 'prompt_tokens': 208, 'total_tokens': 210, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ec08ae704b', 'id': 'chatcmpl-DcoGBt7BL9GcfESq0h7VveSIzYM5X', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e0186-3a2a-7fe3-9ed2-60da15982041-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 208, 'output_tokens': 2, 'total_tokens': 210, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 360, 'prompt_tokens': 196, 'total_tokens': 556, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c34fdfcd04', 'id': 'chatcmpl-DcoGDvTgciIM67VMWFAEjQvq7TflE', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e0186-417c-71f0-a80c-abe9189a86ec-0', tool_calls=[{'name': 'send_mail', 'args': {'receiver_email': 'ukpaajee@gmail.com', 'subject': 'Summary and Translation of AI Text', 'body': 'Summary:\nArtificial intelligence (AI) is a branch of computer science that develops systems capable of tasks that require human intelligence, such as learning, reasoning, and problem-solving. It is a driving force behind modern technology, notably through machine learning and neural networks, and is extensively used in generative tools, predictive analysis, and automation in everyday life and business.\n\nTranslation in Nepali:\nआर्टिफिशियल इन्टेलिजेन्स (एआई) कम्प्युटर विज्ञानको एउटा शाखा हो जसले मानव बुद्धिमत्ताको आवश्यकता पर्ने कार्यहरू गर्ने प्रणालीहरू सिर्जना गर्छ, जस्तै अध्ययन, तर्क, र समस्या समाधान। यसले आधुनिक प्राविधिलाई मेशिन शिक्षण र न्युरल नेटवर्कहरूको माध्यमबाट शक्ति दिन्छ, जुन प्र जनरेटर उपकरण, भविष्यवाणी विश्लेषण, र दैनिक जीवन र व्यवसायमा स्वचालनका लागि व्यापक रूपमा प्रयोग गरिएको छ।\n\nTranslation in Hindi:\nआर्टिफिशियल इंटेलिजेंस (एआई) कंप्यूटर विज्ञान की एक शाखा है जो ऐसे सिस्टम बनाती है जो मानव बुद्धिमत्ता की आवश्यकता वाले कार्य कर सकते हैं, जैसे कि सीखना, तर्क करना और समस्या को हल करना। यह मशीन लर्निंग और न्यूरल नेटवर्क्स के माध्यम से आधुनिक प्रौद्योगिकी को शक्ति देता है, जिसका व्यापक रूप से जनरेटिव टूल्स, भविष्यवाणी विश्लेषण, और दैनिक जीवन और व्यवसाय में स्वचालन के लिए उपयोग किया जाता है.'}, 'id': 'call_7cWQ4hKsgrFTCpwVNTeUxw0B', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 196, 'output_tokens': 360, 'total_tokens': 556, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})], 'interrupt': [Interrupt(value={'action_requests': [{'name': 'send_mail', 'args': {'receiver_email': 'ukpaajee@gmail.com', 'subject': 'Summary and Translation of AI Text', 'body': 'Summary:\nArtificial intelligence (AI) is a branch of computer science that develops systems capable of tasks that require human intelligence, such as learning, reasoning, and problem-solving. It is a driving force behind modern technology, notably through machine learning and neural networks, and is extensively used in generative tools, predictive analysis, and automation in everyday life and business.\n\nTranslation in Nepali:\nआर्टिफिशियल इन्टेलिजेन्स (एआई) कम्प्युटर विज्ञानको एउटा शाखा हो जसले मानव बुद्धिमत्ताको आवश्यकता पर्ने कार्यहरू गर्ने प्रणालीहरू सिर्जना गर्छ, जस्तै अध्ययन, तर्क, र समस्या समाधान। यसले आधुनिक प्राविधिलाई मेशिन शिक्षण र न्युरल नेटवर्कहरूको माध्यमबाट शक्ति दिन्छ, जुन प्र जनरेटर उपकरण, भविष्यवाणी विश्लेषण, र दैनिक जीवन र व्यवसायमा स्वचालनका लागि व्यापक रूपमा प्रयोग गरिएको छ।\n\nTranslation in Hindi:\nआर्टिफिशियल इंटेलिजेंस (एआई) कंप्यूटर विज्ञान की एक शाखा है जो ऐसे सिस्टम बनाती है जो मानव बुद्धिमत्ता की आवश्यकता वाले कार्य कर सकते हैं, जैसे कि सीखना, तर्क करना और समस्या को हल करना। यह मशीन लर्निंग और न्यूरल नेटवर्क्स के माध्यम से आधुनिक प्रौद्योगिकी को शक्ति देता है, जिसका व्यापक रूप से जनरेटिव टूल्स, भविष्यवाणी विश्लेषण, और दैनिक जीवन और व्यवसाय में स्वचालन के लिए उपयोग किया जाता है.'}, 'description': "Tool execution pending approval\n\nTool: send_mail\nArgs: {'receiver_email': 'ukpaajee@gmail.com', 'subject': 'Summary and Translation of AI Text', 'body': 'Summary:\\nArtificial intelligence (AI) is a branch of computer science that develops systems capable of tasks that require human intelligence, such as learning, reasoning, and problem-solving. It is a driving force behind modern technology, notably through machine learning and neural networks, and is extensively used in generative tools, predictive analysis, and automation in everyday life and business.\\n\\nTranslation in Nepali:\\nआर्टिफिशियल इन्टेलिजेन्स (एआई) कम्प्युटर विज्ञानको एउटा शाखा हो जसले मानव बुद्धिमत्ताको आवश्यकता पर्ने कार्यहरू गर्ने प्रणालीहरू सिर्जना गर्छ, जस्तै अध्ययन, तर्क, र समस्या समाधान। यसले आधुनिक प्राविधिलाई मेशिन शिक्षण र न्युरल नेटवर्कहरूको माध्यमबाट शक्ति दिन्छ, जुन प्र जनरेटर उपकरण, भविष्यवाणी विश्लेषण, र दैनिक जीवन र व्यवसायमा स्वचालनका लागि व्यापक रूपमा प्रयोग गरिएको छ।\\n\\nTranslation in Hindi:\\nआर्टिफिशियल इंटेलिजेंस (एआई) कंप्यूटर विज्ञान की एक शाखा है जो ऐसे सिस्टम बनाती है जो मानव बुद्धिमत्ता की आवश्यकता वाले कार्य कर सकते हैं, जैसे कि सीखना, तर्क करना और समस्या को हल करना। यह मशीन लर्निंग और न्यूरल नेटवर्क्स के माध्यम से आधुनिक प्रौद्योगिकी को शक्ति देता है, जिसका व्यापक रूप से जनरेटिव टूल्स, भविष्यवाणी विश्लेषण, और दैनिक जीवन और व्यवसाय में स्वचालन के लिए उपयोग किया जाता है.'}"}], 'review_configs': [{'action_name': 'send_mail', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='0a6e34b13a3a95e77c906e860c46b613')]}

In [ ]:
result

In [ ]:
graph.get_graph()

In [ ]:
graph.get_state(session_id="ajeet")

In [ ]:
from langchain.tools import tool, ToolRuntime
from smtplib import SMTP
from email.mime.text import MIMEText

from agents import Agent, Runner, function_tool


@tool
async def send_mail(receiver_email: str, subject: str, body: str) -> dict:
    """Send an email to receiver_email with the given subject and body."""
    return f"email sent to {receiver_email}, {subject} {body}"

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware 
from langgraph.checkpoint.memory import InMemorySaver 

from langchain.chat_models import init_chat_model

from dotenv import load_dotenv
load_dotenv()

model = init_chat_model("gpt-4o-mini")

email_agent = create_agent(
    model="gpt-4o-mini",
    tools=[send_mail],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_mail": True, 
            },
            description_prefix="Tool execution pending approval",
        ),
    ],
    # system_prompt="You are a email agent. Help with drafting, sending email to the user.",
)


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from langchain_agent.graph import graph as gp

text = """Artificial intelligence (AI) is a branch of computer science that creates systems capable of performing tasks requiring human intelligence, such as learning, reasoning, and problem-solving. It powers modern technology through machine learning and neural networks, widely used for generative tools, predictive analysis, and automation in daily life and business."""
result = await gp.run(
    {
        "messages": [
            {
                "role": "user",
                "content": f"Hi, I want you to summairze the given text and then translate it to nepali and hindi: {text} and then mail the content to the user ukpaajee@gmail.com",
            }
        ]
    },
    session_id="da"
)

after calling the orchestrator agent the output is: email_agent
By orchestrator agent, GOTO: email_agent




StateSnapshot(values={'messages': [HumanMessage(content='Hi, I want you to summairze the given text and then translate it to nepali and hindi: Artificial intelligence (AI) is a branch of computer science that creates systems capable of performing tasks requiring human intelligence, such as learning, reasoning, and problem-solving. It powers modern technology through machine learning and neural networks, widely used for generative tools, predictive analysis, and automation in daily life and business. and then mail the content to the user ukpaajee@gmail.com', additional_kwargs={}, response_metadata={}, id='ff695ebb-02b6-451a-acf5-abd96d79e3d3'), AIMessage(content='email_agent', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 2, 'prompt_tokens': 208, 'total_tokens': 210, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'aud

In [2]:
from langgraph.types import Command

result = await gp.run(
    Command(
        resume={
            "decisions": [
                {
                    "type": "approve",
                }
            ]
        }
    ),
    session_id="da"
)





StateSnapshot(values={'messages': [HumanMessage(content='Hi, I want you to summairze the given text and then translate it to nepali and hindi: Artificial intelligence (AI) is a branch of computer science that creates systems capable of performing tasks requiring human intelligence, such as learning, reasoning, and problem-solving. It powers modern technology through machine learning and neural networks, widely used for generative tools, predictive analysis, and automation in daily life and business. and then mail the content to the user ukpaajee@gmail.com', additional_kwargs={}, response_metadata={}, id='ff695ebb-02b6-451a-acf5-abd96d79e3d3'), AIMessage(content='email_agent', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 2, 'prompt_tokens': 208, 'total_tokens': 210, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_

In [4]:
result.value['messages']

[HumanMessage(content='Hi, I want you to summairze the given text and then translate it to nepali and hindi: Artificial intelligence (AI) is a branch of computer science that creates systems capable of performing tasks requiring human intelligence, such as learning, reasoning, and problem-solving. It powers modern technology through machine learning and neural networks, widely used for generative tools, predictive analysis, and automation in daily life and business. and then mail the content to the user ukpaajee@gmail.com', additional_kwargs={}, response_metadata={}, id='ff695ebb-02b6-451a-acf5-abd96d79e3d3'),
 AIMessage(content='email_agent', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 2, 'prompt_tokens': 208, 'total_tokens': 210, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'mo